<font color='darkred'> Unless otherwise noted, **this notebook will not be reviewed or autograded.**</font> You are welcome to use it for scratchwork, but **only the files listed in the exercises will be checked.**

---

# Exercises

For these exercises, you'll be creating a [Python class](https://www.geeksforgeeks.org/python/python-classes-and-objects/) in the *apputil.py* file that uses an environment variable assigned in a separate `.env` file.

A few notes:

- Some of these exercises will require some "JSON digging".
- Your functions should still adhere to [best practices](https://codesignal.com/learn/courses/clean-code-basics-with-python/lessons/clean-function-design-in-python), so **your class should probably contain more functions than only the ones dictated below**.
- Since your .env file should be ignored in your .gitignore file, **you will need to use the "Upload" option for this week's autograder.**

## Exercise 1

1. First, create a `.env` file that declares your access token as a variable. Make sure that the `.env` file is ignored as a line in your *.gitignore* file.
2. Then, create a Python class named `Genius` such that the following code initializes the object using the environment variable defined in \#1. You'll use this instantiated class for Exercises 2 and 3.

```python
from apputil import Genius
from dotenv import load_dotenv

load_dotenv()

# constants
ACCESS_TOKEN = os.environ['ACCESS_TOKEN']

genius = Genius(access_token=ACCESS_TOKEN)
```

In [14]:
import os
from dotenv import load_dotenv
from apputil import Genius

load_dotenv()

ACCESS_TOKEN = os.environ["ACCESS_TOKEN"]

genius = Genius(access_token=ACCESS_TOKEN)

In [1]:
class Genius:
    """
    A class for interacting with the Genius API.
    """

    def __init__(self, access_token):
        self.access_token = access_token

## Exercise 2

Reference the `json_data` in the lab. Notice, when we search for an artist name (e.g., "Missy Elliot") in Genius, the result is a list of *songs* attributed to that artist. Suppose we want to capture information about the artist themselves.

Create a method for our `Genius` class called `.get_artist(search_term)` which does the following:

1. Extract the (most likely, "Primary") Artist ID from the first "hit" of the `search_term`.
2. Use the [API path](https://docs.genius.com/#artists-h2) (under "Artists") for this Artist ID to pull information about the artist.
3. **Return** the dictionary containing the resulting JSON object.

For example, the following code should return a dictionary of artist information:

```python
genius.get_artist("Radiohead")
```

In [16]:
import requests


class Genius:
    BASE_URL = "https://api.genius.com"

    def __init__(self, access_token):
        self.access_token = access_token
        self.headers = {
            "Authorization": f"Bearer {self.access_token}"
        }

    def _get(self, endpoint, params=None):
        response = requests.get(
            f"{self.BASE_URL}{endpoint}",
            headers=self.headers,
            params=params
        )
        response.raise_for_status()
        return response.json()

    def get_artist(self, search_term):
        # Search Genius
        search_data = self._get(
            "/search",
            params={"q": search_term}
        )

        # Artist ID from first hit
        artist_id = search_data["response"]["hits"][0]["result"]["primary_artist"]["id"]

        # Get artist info
        artist_data = self._get(f"/artists/{artist_id}")

        # Return the artist JSON dictionary
        return artist_data

## Exercise 3

Use the result from Exercise 2 to create another method for our `Genius` class called `.get_artists(search_terms)` (plural) which takes in a *list* of search terms, and returns a DataFrame containing a row for each search term, and the following columns:

- `search_term`: the raw search term from `search_terms`
- `artist_name`: the (most likely) artist name for the search term
- `artist_id`: the Genius Artist ID for that artist, based on the API call
- `followers_count`: the number of followers for that artist (if available)

For example, the following should return a DataFrame with 4 rows:

```python
genius.get_artists(['Rihanna', 'Tycho', 'Seal', 'U2'])
```

In [12]:
from dotenv import load_dotenv
import os
from apputil import Genius

load_dotenv()

ACCESS_TOKEN = os.getenv("ACCESS_TOKEN")

genius = Genius(access_token=ACCESS_TOKEN)

df = genius.get_artists([
    "Rihanna",
    "Tycho",
    "Seal",
    "U2"
])

print(df)

Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3


HTTPError: 401 Client Error: Unauthorized for url: https://api.genius.com/search?q=Rihanna

## Bonus Exercise (optional)

1. Gather a list of 100+ various musical artists, and save this list in a TXT file.
2. Write a Python *script* that saves the result of `.get_artists` for these artists in a CSV file.
3. If you have time, adjust this script to use multiprocessing.

In [4]:
from dotenv import load_dotenv
import os

In [9]:
def load_artists(filename):
    with open(filename, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def main():
    load_dotenv()

    access_token = os.getenv("ACCESS_TOKEN")

    genius = Genius(access_token=access_token)

    artists = load_artists("artists.txt")

    df = genius.get_artists(artists)

    df.to_csv("artists.csv", index=False)

    print(f"Saved {len(df)} artists to artists.csv")

